# ForestWatch Papua — Improve Model v2 (Fine-tuning Kelas Minoritas)

**Tujuan**: naikkan IoU 5 kelas lemah (Lahan Terbuka, Sawit, Pertanian Lain, Tambang,
Permukiman) **tanpa menurunkan** 2 kelas mayoritas (Perairan, Hutan), di atas checkpoint
**model_1 (Attention U-Net)** yang sudah dilatih — **tanpa train ulang dari nol**.

**Dataset**: `Bahan_Training_Fix_Combined_v4` (lihat `export_gel7_train_valtest.ipynb` bagian T8)
— Papua lama + gelombang-4 (train/val/test, split per-region aman leakage) + gelombang-5 + -6 +
**gelombang-7** (train + val_new7/test_new7). TRAIN kini **~29% piksel kelas lemah** (jauh lebih
kaya dari sebelumnya). val/test diperbesar via gel.7 (region BARU, anti-leakage per-region).

## Kenapa berubah dari v1 (penting)
Run v1 (cRT encoder-**beku** + sampler class-balanced saja, loss **unweighted**, 20 epoch, v3)
**gagal menaikkan mIoU** (TEST mIoU 0,5246 → 0,5286; Lahan Terbuka malah turun). Confusion matrix:
model **collapse ke Hutan**. Dua sebab: (1) encoder beku → fitur kelas lemah tak bisa diperbaiki;
(2) loss tanpa bobot → tak ada tekanan melawan dominasi Hutan. Decoupling/cRT (Kang dkk. 2020)
memang untuk situasi **data tetap** — sekarang data sudah ditambah banyak (gel.7 → v4), jadi
premisnya tak berlaku lagi.

## Metode v2 (strategi baru)
- **Full fine-tune — encoder ResNet50 DIBUKA**, dengan **LR bertingkat** (encoder `1e-5`,
  decoder+head `1e-4`): *discriminative fine-tuning* (Howard & Ruder, ACL 2018) — lapisan dalam
  (fitur umum) di-tune jauh lebih pelan agar tak merusak fitur ImageNet, lapisan atas (spesifik
  tugas) lebih cepat. Ini memungkinkan fitur kelas lemah benar-benar berubah.
- **Loss berbobot**: `0.3·Focal + 0.3·Tversky(β=0,7>α=0,3) + 0.4·CE(weighted)`. Bobot kelas =
  **median-frequency** (Eigen & Fergus, ICCV 2015; justifikasi long-tail: Cui dkk., CVPR 2019).
  Porsi CE dinaikkan ke 0,4 agar bobot benar-benar menggigit. Tversky β>α menekan false-negative
  → recall minoritas naik (Salehi 2017; Abraham & Khan, ISBI 2019); Focal utk contoh sulit
  (Lin dkk., ICCV 2017).
- **Sampler frequency-proportional** (bukan class-balanced murni): oversample patch kelas langka
  secara proporsional. Dipasang dgn weighted loss yang sudah kuat — sengaja sampler lebih lembut
  agar tak **over-correct** kelas mayoritas (Zhang dkk., *Bag of Tricks for Long-Tailed*, 2021).
- **Seleksi checkpoint terjaga (guarded)**: epoch disimpan hanya bila IoU mayoritas (VAL) tak
  turun > ε (ε=0,01) dari baseline → mayoritas dijamin tak dikorbankan.
- **Anti-bocor**: seleksi pakai VAL; TEST sekali di akhir. **Reversible**: hasil →
  `best_model_finetune_v2.pt`; `best_model.pt` baseline tak disentuh. **40 epoch**, resume-safe.

## ⚠️ Catatan jujur
Config ini **lebih mungkin** menaikkan mIoU daripada v1 (memperbaiki 2 sebab kegagalan + data
jauh lebih kaya), **tetapi tidak ada jaminan** menyentuh angka tertentu (mis. 0,8) — itu di luar
kendali config semata. Angka final tetap diuji jujur di TEST (sekali).

**Referensi**: Howard & Ruder ACL 2018 (discriminative fine-tuning); Kang dkk. ICLR 2020
(decoupling/cRT — alasan ditinggalkan saat data nambah); Eigen & Fergus ICCV 2015 (median-freq);
Cui dkk. CVPR 2019 (class-balanced loss); Lin dkk. ICCV 2017 (Focal); Salehi 2017 & Abraham &
Khan ISBI 2019 (Tversky/Focal-Tversky); Zhang dkk. 2021 (bag of tricks long-tailed); Oktay dkk.
MICCAI 2018 (attention-gate, dasar pilih model_1).


## Bagian 0 — Setup environment (Colab / Lab / Kaggle)


In [ ]:
# === Bagian 0 — Setup (set ENV = "colab" / "lab" / "kaggle") ===
ENV = "colab"   # "colab" | "lab" (PC+Drive Desktop) | "kaggle"
import os, sys, subprocess, importlib
from pathlib import Path

# Auto-deteksi Kaggle (folder /kaggle hanya ada di runtime Kaggle).
if Path("/kaggle").exists() and ENV != "kaggle":
    print(f"[auto-detect] /kaggle -> override ENV='{ENV}' -> 'kaggle'")
    ENV = "kaggle"

DRIVE_ROOT = None
_SRC = None
if ENV == "colab":
    subprocess.run("cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
                   "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
                   shell=True, check=False)
    subprocess.run("pip install -q -e /content/fw_repo[ml]", shell=True, check=False)
    _SRC = "/content/fw_repo/model/src"
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Satria Data 3.0")
elif ENV == "lab":
    DRIVE_ROOT = Path(r"G:/My Drive/Satria Data 3.0")   # <- SESUAIKAN mount Drive Desktop lab
elif ENV == "kaggle":
    subprocess.run("cd /kaggle/working && (git -C fw_repo pull -q || git clone --depth 1 "
                   "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
                   shell=True, check=False)
    subprocess.run("pip install -q --no-deps -e /kaggle/working/fw_repo[ml]", shell=True, check=False)
    subprocess.run("pip install -q --no-deps segmentation-models-pytorch albumentations torchmetrics",
                   shell=True, check=False)
    _SRC = "/kaggle/working/fw_repo/model/src"
else:
    raise ValueError("ENV harus 'colab' | 'lab' | 'kaggle'")

if _SRC and _SRC not in sys.path:
    sys.path.insert(0, _SRC)
for _m in [m for m in list(sys.modules) if m == "forestwatch" or m.startswith("forestwatch.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

import torch
_gpu = f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""
print(f"ENV={ENV} | CUDA={torch.cuda.is_available()}{_gpu}")


In [ ]:
# === Deklarasi path, identitas model, & config ===
from forestwatch.config import load_config
from forestwatch.utils.io import save_json, load_json
from forestwatch.constants import N_CLASSES, CLASS_NAMES, CLASS_COLORS
cfg = load_config()

MODEL_KEY  = "model_1_attention_unet"   # pemenang banding (compare_and_select_best_model.ipynb)
MODEL_ARCH = dict(architecture="unet_scse", encoder_name="resnet50")
STRONG = [0, 1]            # Perairan, Hutan -- JAGA (jangan turun)
WEAK   = [2, 3, 4, 5, 6]   # Lahan Terbuka, Sawit, Pertanian Lain, Tambang, Permukiman -- target naik

# Folder BARU khusus hasil fine-tune (terpisah dari Model_Comparison/<model_key> yg isinya
# 3 model baseline) -- sejajar (sibling) di ForestWatch_Outputs, gampang dicari.
# DRIVE_TARGET_NAME dipakai juga di cell promosi (kaggle) sbg nama folder tujuan unggah manual.
DRIVE_TARGET_NAME = f"ForestWatch_Outputs/Model_Improve/{MODEL_KEY}"

if ENV in ("colab", "lab"):
    # v4 (lihat export_gel7_train_valtest.ipynb bagian T8) = Papua lama + gel.4 (train/val/test,
    # split per-region aman dari leakage) + gel.5 + gel.6 + gel.7 (train + val_new7/test_new7,
    # 5 kelas lemah, gel.7 menambah ~26rb patch train) -- gantikan v3. Jangan pakai
    # "Bahan_Training_Fix" (Papua-only) lagi di sini.
    BAHAN_DIR   = DRIVE_ROOT / "Bahan_Training_Fix_Combined_v4"
    MODELS_ROOT = DRIVE_ROOT / "ForestWatch_Outputs" / "Model_Comparison"   # baseline 3 model (READ-ONLY di sini)
    BASE_DIR    = MODELS_ROOT / MODEL_KEY
    BASE_CKPT   = BASE_DIR / "best_model.pt"
    FT_DIR      = DRIVE_ROOT / "ForestWatch_Outputs" / "Model_Improve" / MODEL_KEY   # folder BARU
else:  # kaggle: data + checkpoint baseline dari dataset yg di-attach (Add Input)
    # Attach dataset Bahan_Training_Fix_Combined_v4 (gel.7; bukan v1/v2/v3 lama) sbg Kaggle Input.
    BAHAN_DIR = None                       # diisi di cell ekstrak (rglob)
    BASE_DIR  = None
    BASE_CKPT = next(Path("/kaggle/input").rglob("best_model.pt"))
    FT_DIR    = Path("/kaggle/working") / "Model_Improve" / MODEL_KEY   # lokal sesi; lihat cell promosi utk unduh

FT_DIR.mkdir(parents=True, exist_ok=True)
FT_CKPT    = FT_DIR / "best_model_finetune_v2.pt"        # v2 = full FT (encoder dibuka) + weighted loss, dataset v4
FT_RESUME  = FT_DIR / "best_model_finetune_resume_v2.pt" # nama baru: state run lama (param-group beda) tak kompatibel
FT_SAMPLER = FT_DIR / "patch_sampler_v2.json"           # cache bobot sampler frequency-proportional (skema baru)
OUT_DIR    = FT_DIR

assert BASE_CKPT.exists(), f"Baseline checkpoint tak ada: {BASE_CKPT}"
print("Baseline ckpt :", BASE_CKPT)
print("Dataset       :", BAHAN_DIR if BAHAN_DIR else "(kaggle, attach Bahan_Training_Fix_Combined_v3)")
print("Output FT dir :", FT_DIR)
print("Kelas JAGA    :", [CLASS_NAMES[c] for c in STRONG])
print("Kelas target  :", [CLASS_NAMES[c] for c in WEAK])


In [ ]:
# === Ekstrak dataset ke disk lokal + daftar file (train/val/test) ===
# Pola sama spt notebook training: extract .tar -> disk lokal (lepas dari bottleneck Drive FUSE).
# BAHAN_DIR sudah menunjuk ke Bahan_Training_Fix_Combined_v3 (cell sebelumnya) -- train/val/test
# di sini SUDAH gabungan Papua lama + gel.4 (train/val/test) + gel.5 + gel.6 (train-only).
# Tidak perlu merge manual lagi spt revisi sebelumnya (cell itu sudah dihapus -- akan
# double-count gel.4 val/test kalau dijalankan lagi di atas v3).
from forestwatch.data.dataset import extract_dataset_archives
from forestwatch.data.patches import list_patches

if ENV in ("colab", "lab"):
    LOCAL_DIR = Path("/content/dataset_local") if ENV == "colab" else Path.home() / "dataset_local"
    _splits = ("train", "val", "test")
    if (BAHAN_DIR / "train_rajaampat").exists():
        _splits = _splits + ("train_rajaampat",)
    local_dirs = extract_dataset_archives(BAHAN_DIR, LOCAL_DIR, splits=_splits, max_workers=8)
    cw_path = BAHAN_DIR / "class_weights.json"
else:  # kaggle
    # GROUND TRUTH (dicek manual via UI Kaggle): Kaggle AUTO-EKSTRAK semua .tar yg diupload,
    # rekursif, sampai habis -- jadi di /kaggle/input SUDAH .npz mentah, BUKAN .tar lagi.
    #   - train (fw-papua-train-1/-2, diupload --dir-mode tar): muncul sbg subfolder
    #     "train_part01/", "train_part02/", dst, isinya .npz langsung.
    #   - val/test (diupload flat, 1 tar di root): Kaggle extract jadi .npz LANGSUNG di root
    #     dataset, TANPA nama folder yg menyebut "val"/"test" sama sekali -- satu2nya penanda
    #     splitnya adalah nama dataset Kaggle itu sendiri (slug "fw-papua-val"/"fw-papua-test").
    BAHAN_SRC = Path("/kaggle/temp/bahan_src"); BAHAN_SRC.mkdir(parents=True, exist_ok=True)
    for s in ("train", "val", "test", "train_rajaampat"):
        (BAHAN_SRC / s).mkdir(parents=True, exist_ok=True)

    for npz in Path("/kaggle/input").rglob("*.npz"):
        parts_lower = [p.lower() for p in npz.parts]
        path_str = str(npz).lower()
        if any(p.startswith("train_part") for p in parts_lower):
            split = "train"
        elif "rajaampat" in path_str:
            split = "train_rajaampat"
        elif "fw-papua-val" in path_str or any(p == "val" for p in parts_lower):
            split = "val"
        elif "fw-papua-test" in path_str or any(p == "test" for p in parts_lower):
            split = "test"
        else:
            continue
        _dst = BAHAN_SRC / split / npz.name
        if not _dst.exists():
            os.symlink(npz, _dst)

    for s in ("train", "val", "test", "train_rajaampat"):
        n = len(list((BAHAN_SRC / s).glob("*.npz")))
        if n:
            print(f"{s}: {n} file .npz ditemukan -> {BAHAN_SRC / s}")

    _splits = tuple(s for s in ("train", "val", "test", "train_rajaampat")
                     if next((BAHAN_SRC / s).glob("*.npz"), None) is not None)
    local_dirs = {s: BAHAN_SRC / s for s in _splits}
    cw_path = BAHAN_SRC / "class_weights.json"

final_train_files = list_patches(local_dirs["train"])
if "train_rajaampat" in local_dirs:
    _ra = list_patches(local_dirs["train_rajaampat"])
    final_train_files = final_train_files + _ra
    print(f"  + {len(_ra)} patch Raja Ampat (Tambang asli)")
val_p  = list_patches(local_dirs["val"])
test_p = list_patches(local_dirs["test"])
assert final_train_files and val_p and test_p, "train/val/test kosong -- cek BAHAN_DIR / attach dataset."
print(f"train={len(final_train_files)} | val={len(val_p)} | test={len(test_p)}")


In [ ]:
# === Distribusi kelas TRAIN + bobot median-frequency (Eigen & Fergus 2015) ===
# class_w dipakai DUA-DUANYA: (1) sampler frequency-proportional (cell berikut) +
# (2) komponen CE di loss (cell loss). Median-frequency: w(c) = median(freq) / freq(c) --
# kelas langka (Tambang) dapat bobot besar, Hutan dapat bobot kecil; di-clip [0.3, 5.0]
# supaya training stabil. Justifikasi long-tail: Cui dkk. CVPR 2019 (Class-Balanced Loss).
import json
from forestwatch.data.patches import compute_class_distribution
from forestwatch.training.metrics import median_frequency_weights

_dist_cache = FT_DIR / "train_distribution.json"
if _dist_cache.exists():
    train_dist = {int(k): int(v) for k, v in json.load(open(_dist_cache)).items()}
    print("Distribusi train dimuat dari cache:", _dist_cache.name)
else:
    # Hitung penuh sekali (~beberapa menit utk puluhan ribu patch) lalu cache.
    train_dist = compute_class_distribution(local_dirs["train"], n_classes=N_CLASSES)
    save_json({str(k): int(v) for k, v in train_dist.items()}, _dist_cache)

class_w = median_frequency_weights(train_dist, n_classes=N_CLASSES)

_tot = sum(train_dist.values()) or 1
print(f"{'kelas':<16}{'piksel':>16}{'freq%':>9}{'bobot':>9}")
for c in range(N_CLASSES):
    n = train_dist.get(c, 0)
    print(f"{CLASS_NAMES[c]:<16}{n:>16,}{100*n/_tot:>8.2f}%{class_w[c]:>9.3f}")
print("\nclass_w (median-frequency) ->", [round(w, 3) for w in class_w])
print("Dipakai utk: sampler frequency-proportional + komponen CE di loss.")


In [ ]:
import numpy as np
# === Sampler: frequency-proportional (oversample kelas langka, BUKAN class-balanced murni) ===
# Dipasangkan dgn weighted loss (median-frequency) -> sengaja pilih sampler yg LEBIH LEMBUT
# daripada class-balanced murni (1/7 equal): menumpuk 2 rebalancer agresif (sampler CB +
# loss reweight kuat) bisa over-correct & menurunkan kelas mayoritas (Zhang dkk. 2021,
# "Bag of Tricks for Long-Tailed"). Frequency-proportional: bobot patch = Sum_c class_w[c] *
# fraksi piksel kelas c -- patch berisi kelas langka lebih sering ter-sampling, proporsional.
from forestwatch.data.dataset import compute_patch_sampler_weights

train_sampler_weights = compute_patch_sampler_weights(
    final_train_files, class_w, n_classes=N_CLASSES, cache_path=FT_SAMPLER,
)
_w = np.array(train_sampler_weights)
print(f"Bobot sampler: n={len(_w):,} | min={_w.min():.3e} | median={np.median(_w):.3e} | max={_w.max():.3e}")
print("Sumber bobot: class_w (median-frequency) x fraksi piksel kelas per-patch.")


In [ ]:
# === Build DataLoaders (train: sampler frequency-proportional) + model (encoder DIBUKA) + loss berbobot ===
from forestwatch.data.dataset import PapuaDataset
from forestwatch.model.architecture import build_unet, count_parameters
from forestwatch.model.losses import make_loss_fn
from forestwatch.training.trainer import set_encoder_trainable
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch

N_WORKERS = 2 if ENV == "colab" else min(8, (os.cpu_count() or 2))

# Train loader dibangun MANUAL (bukan build_dataloaders_from_files): pakai bobot sampler
# frequency-proportional (cell sebelumnya, dari class_w median-frequency).
train_ds = PapuaDataset(final_train_files, train=True, augment_p=cfg["training"]["augmentation"])
sampler = WeightedRandomSampler(
    weights=train_sampler_weights, num_samples=len(final_train_files), replacement=True,
)
train_loader = DataLoader(
    train_ds, batch_size=cfg["training"]["batch_size"], sampler=sampler,
    num_workers=N_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=(N_WORKERS > 0),
)
val_loader = DataLoader(
    PapuaDataset(val_p, train=False), batch_size=cfg["training"]["batch_size"],
    shuffle=False, num_workers=N_WORKERS, pin_memory=True,
    persistent_workers=(N_WORKERS > 0),
)
print(f"Train {len(train_loader.dataset)} | Val {len(val_loader.dataset)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_unet(in_channels=cfg["model"]["in_channels"], classes=cfg["model"]["classes"],
                   encoder_weights=cfg["model"]["encoder_weights"], **MODEL_ARCH)
model.load_state_dict(torch.load(BASE_CKPT, map_location="cpu"))
model = model.to(device)

# FULL FINE-TUNE: encoder DIBUKA (v4 sudah kaya data kelas lemah -> end-to-end retraining
# lebih kuat drpd decoupling/cRT yg utk data tetap; Kang dkk. ICLR 2020). LR encoder dibuat
# jauh lebih kecil drpd decoder/head di cell loop (discriminative fine-tuning, Howard & Ruder 2018).
assert set_encoder_trainable(model, True), "Gagal buka encoder (model tak punya .encoder?)."
n_train = count_parameters(model)
n_total = sum(p.numel() for p in model.parameters())
print(f"Param trainable (encoder+decoder+head): {n_train:,} / total {n_total:,} "
      f"({100 * n_train / n_total:.1f}%) -- encoder DIBUKA (LR bertingkat di loop)")

# Loss BERBOBOT: class_w (median-frequency) -> komponen CE; porsi CE dinaikkan ke 0.4 agar
# bobot benar2 menggigit (di make_loss_fn, class_weights hanya kena CE). Tversky beta>alpha
# tetap mengangkat recall kelas minoritas (Salehi 2017; Abraham & Khan 2019), Focal utk
# contoh sulit (Lin 2017). Kombinasi: 0.3*Focal + 0.3*Tversky + 0.4*CE(weighted).
_lc = cfg["training"]["loss"]
loss_fn = make_loss_fn(
    components=[("focal", 0.3), ("tversky", 0.3), ("ce", 0.4)],
    class_weights=class_w,
    tversky_alpha=_lc.get("tversky_alpha", 0.3),
    tversky_beta=_lc.get("tversky_beta", 0.7),
    focal_gamma=_lc.get("focal_gamma", 2.0),
    device=device,
)
print("Loss: 0.3*Focal + 0.3*Tversky(beta>alpha) + 0.4*CE(weighted median-frequency).")
print("Rebalance: weighted loss (CE) + sampler frequency-proportional (keduanya dari class_w).")


## Baseline (BEFORE) + tetapkan floor kelas mayoritas (anti-bocor: VAL utk guard, TEST utk laporan)


In [ ]:
# === Baseline (BEFORE) per-kelas IoU: VAL (utk guard) + TEST (utk laporan akhir) ===
# Loop manual baca .npz satu-satu (TANPA DataLoader worker) -> RAM stabil (versi DataLoader
# terbukti bocor puluhan GB di lingkungan ini). VAL menetapkan 'floor' kelas mayoritas;
# TEST HANYA laporan -- JANGAN dipakai utk seleksi checkpoint (cegah kebocoran).
import numpy as np, torch, gc
from forestwatch.training.metrics import compute_confusion_matrix, metric_summary

def _eval_files(mdl, files):
    mdl.eval()
    cm = np.zeros((N_CLASSES, N_CLASSES), dtype=np.int64)
    with torch.no_grad():
        for i, fp in enumerate(files):
            d = np.load(fp)
            img = torch.from_numpy(d["img"]).float().unsqueeze(0).to(device)
            pr = mdl(img).argmax(1).squeeze(0).cpu().numpy().astype(np.uint8)
            cm += compute_confusion_matrix(pr, d["lab"], n_classes=N_CLASSES)
            del d, img, pr
            if i % 3000 == 0:
                gc.collect(); torch.cuda.empty_cache()
    return metric_summary(cm, class_names=CLASS_NAMES)

base_val  = _eval_files(model, val_p)
base_test = _eval_files(model, test_p)
base_val_iou  = [r["iou"] for r in base_val["per_class"]]
base_test_iou = [r["iou"] for r in base_test["per_class"]]

EPS = 0.01    # toleransi penurunan kelas mayoritas (guard); dilonggarkan dari 0.005 krn
              # full fine-tune (encoder dibuka) bisa buat mayoritas ~0.95-0.99 sedikit goyang;
              # 0.01 masih proteksi kuat tapi tak menolak checkpoint yg menaikkan mIoU keseluruhan.
floor = {c: base_val_iou[c] - EPS for c in STRONG}
print(f"Baseline VAL mIoU={base_val['mean_iou']:.4f} | TEST mIoU={base_test['mean_iou']:.4f} "
      f"| TEST FWIoU={base_test['fwiou']:.4f}")
print("Floor mayoritas (VAL):", {CLASS_NAMES[c]: round(floor[c], 4) for c in STRONG})
print(f"\n{'kelas':<16}{'VAL IoU':>9}{'TEST IoU':>10}")
for c in range(N_CLASSES):
    print(f"{CLASS_NAMES[c]:<16}{base_val_iou[c]:>9.4f}{base_test_iou[c]:>10.4f}")


## Fine-tune (encoder DIBUKA, LR bertingkat) — seleksi checkpoint terjaga + resume-safe


In [ ]:
# === Fine-tune (encoder BEKU) -- seleksi checkpoint TERJAGA (guarded) ===
# Epoch jadi 'best' HANYA bila IoU kelas mayoritas (VAL) >= floor; di antara yg lolos, ambil
# val mIoU tertinggi -> kelas mayoritas dijamin tak turun. Resume-safe (anti sesi Colab mati).
import time, torch
from torch.amp import autocast
from torchmetrics.classification import MulticlassJaccardIndex
from tqdm.auto import tqdm
from forestwatch.training.metrics import set_seed
try:
    from torch.amp import GradScaler; _NEWSCALER = True      # torch >= 2.4
except ImportError:
    from torch.cuda.amp import GradScaler; _NEWSCALER = False  # torch < 2.4 (Jetson)

# LR bertingkat (discriminative fine-tuning, Howard & Ruder 2018): encoder (fitur umum) di-tune
# jauh lebih pelan drpd decoder+head. Cegah catastrophic-forgetting fitur ImageNet.
FT_LR_ENC, FT_LR_HEAD = 1e-5, 1e-4; FT_EPOCHS = 40; FT_PATIENCE = 12
set_seed(cfg["project"]["seed"])
use_amp = torch.cuda.is_available()

enc_params  = [p for n, p in model.named_parameters()
               if n.startswith("encoder.") and p.requires_grad]
head_params = [p for n, p in model.named_parameters()
               if not n.startswith("encoder.") and p.requires_grad]
assert enc_params and head_params, "param-group kosong (encoder/head) -- cek nama parameter model"
trainable = enc_params + head_params
optimizer = torch.optim.AdamW(
    [{"params": enc_params, "lr": FT_LR_ENC},
     {"params": head_params, "lr": FT_LR_HEAD}],
    weight_decay=cfg["training"]["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FT_EPOCHS)
print(f"Optimizer: AdamW 2 grup -- encoder lr={FT_LR_ENC:.0e} ({len(enc_params)} tensor), "
      f"head lr={FT_LR_HEAD:.0e} ({len(head_params)} tensor) | {FT_EPOCHS} epoch, patience {FT_PATIENCE}")
scaler = (GradScaler(device.type) if _NEWSCALER else GradScaler()) if use_amp else None
iou_pc = MulticlassJaccardIndex(num_classes=N_CLASSES, average=None).to(device)

start_epoch, best_obj, best_epoch, wait, history = 1, -1.0, -1, 0, []
if FT_RESUME.exists():
    try:
        st = torch.load(FT_RESUME, map_location=device)
        model.load_state_dict(st["model"]); optimizer.load_state_dict(st["optimizer"])
        scheduler.load_state_dict(st["scheduler"])
        if scaler and st.get("scaler"): scaler.load_state_dict(st["scaler"])
        start_epoch = st["epoch"] + 1; best_obj = st["best_obj"]; best_epoch = st["best_epoch"]
        wait = st["wait"]; history = st["history"]
        set_encoder_trainable(model, True)   # pastikan encoder tetap TERBUKA pasca-resume
        print(f"Resume -> epoch {start_epoch} (best guarded val mIoU={best_obj:.4f})")
    except Exception as e:
        print("Gagal resume:", e)

for ep in range(start_epoch, FT_EPOCHS + 1):
    model.train()   # encoder ikut belajar (dibuka); BN encoder update normal
    tr = 0.0; t0 = time.time()
    for x, y in tqdm(train_loader, desc=f"ep{ep:02d} train", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with autocast(device_type=device.type):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss = loss_fn(model(x), y); loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0); optimizer.step()
        tr += float(loss.item())
    scheduler.step(); tr /= max(len(train_loader), 1)

    model.eval(); iou_pc.reset(); vl = 0.0
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc=f"ep{ep:02d} val", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            p = model(x); vl += float(loss_fn(p, y).item()); iou_pc.update(p.argmax(1), y)
    vl /= max(len(val_loader), 1)
    pc = [float(v) for v in iou_pc.compute().tolist()]
    vmiou = sum(pc) / len(pc); weak_miou = sum(pc[c] for c in WEAK) / len(WEAK)
    strong_ok = all(pc[c] >= floor[c] for c in STRONG)
    history.append({"epoch": ep, "train_loss": tr, "val_loss": vl, "val_miou": vmiou,
                    "weak_miou": weak_miou, "val_iou_per_class": [round(v, 4) for v in pc],
                    "strong_ok": bool(strong_ok), "lr": optimizer.param_groups[0]["lr"],
                    "epoch_time_sec": time.time() - t0})
    print(f"ep{ep:02d} | loss {tr:.4f} | val {vl:.4f} | mIoU {vmiou:.4f} | "
          f"weak {weak_miou:.4f} | mayoritas_ok={strong_ok}")
    print("   IoU/kelas:", [round(v, 3) for v in pc])

    if strong_ok and vmiou > best_obj:
        best_obj, best_epoch, wait = vmiou, ep, 0
        torch.save(model.state_dict(), FT_CKPT)
        print(f"   -> best guarded checkpoint (val mIoU={vmiou:.4f}) -> {FT_CKPT.name}")
    else:
        wait += 1
    torch.save({"epoch": ep, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict() if scaler else None,
                "best_obj": best_obj, "best_epoch": best_epoch, "wait": wait,
                "history": history}, FT_RESUME)
    if wait >= FT_PATIENCE:
        print(f"Early stopping di epoch {ep} (patience={FT_PATIENCE})."); break

print(f"\nSelesai. best guarded val mIoU={best_obj:.4f} @ epoch {best_epoch}")
if best_epoch < 0:
    print("PERINGATAN: TAK ada epoch yg lolos guard (semua menurunkan kelas mayoritas).")
    print("-> FT_CKPT tidak tersimpan. Pertahankan baseline; coba turunkan BOOST/FT_LR. JANGAN promosikan.")


In [ ]:
# === Plot kurva fine-tune -> FT_DIR ===
import matplotlib.pyplot as plt
assert history, "history kosong -- jalankan cell fine-tune dulu."
eps = [h["epoch"] for h in history]
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].plot(eps, [h["train_loss"] for h in history], label="train", lw=2)
ax[0].plot(eps, [h["val_loss"] for h in history], label="val", lw=2)
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(eps, [h["val_miou"] for h in history], color="green", lw=2, label="val mIoU")
ax[1].plot(eps, [h["weak_miou"] for h in history], color="red", lw=2, label="weak mIoU")
ax[1].axhline(base_val["mean_iou"], color="gray", ls="--", label="baseline mIoU")
ax[1].set_title("mIoU (val)"); ax[1].set_ylim(0, 1); ax[1].legend(); ax[1].grid(alpha=0.3)
for c in range(N_CLASSES):
    ax[2].plot(eps, [h["val_iou_per_class"][c] for h in history],
               color=CLASS_COLORS[c], lw=1.6, label=CLASS_NAMES[c])
ax[2].set_title("val IoU per-kelas"); ax[2].set_ylim(0, 1)
ax[2].legend(fontsize=7, ncol=2); ax[2].grid(alpha=0.3)
fig.suptitle(MODEL_KEY + " (fine-tune)"); fig.tight_layout()
fig.savefig(FT_DIR / "training_curve_finetune.png", dpi=120, bbox_inches="tight"); plt.show()
print("Disimpan:", FT_DIR / "training_curve_finetune.png")


## Evaluasi akhir di TEST (sekali) — tabel BEFORE/AFTER + vonis jujur


In [ ]:
# === Evaluasi TEST akhir (sekali) + tabel BEFORE/AFTER + vonis jujur ===
import numpy as np, torch
import matplotlib.pyplot as plt
from forestwatch.model.architecture import build_unet, export_to_onnx

if not FT_CKPT.exists():
    print("FT_CKPT tak ada -> fine-tune tak menghasilkan model yg lolos guard.")
    print("VONIS: pertahankan BASELINE. Tidak ada artefak fine-tune utk dipromosikan.")
else:
    ft_model = build_unet(in_channels=cfg["model"]["in_channels"], classes=cfg["model"]["classes"],
                          encoder_weights=cfg["model"]["encoder_weights"], **MODEL_ARCH).to(device)
    ft_model.load_state_dict(torch.load(FT_CKPT, map_location="cpu"))
    ft_test = _eval_files(ft_model, test_p)
    ft_iou = [r["iou"] for r in ft_test["per_class"]]

    print(f"{'kelas':<16}{'BEFORE':>9}{'AFTER':>9}{'delta':>9}")
    drop_majority = []
    for c in range(N_CLASSES):
        dlt = ft_iou[c] - base_test_iou[c]
        flag = ""
        if c in STRONG and dlt < -EPS:
            flag = "  <- MAYORITAS TURUN"; drop_majority.append(c)
        elif c in WEAK and dlt > 0:
            flag = "  <- naik"
        print(f"{CLASS_NAMES[c]:<16}{base_test_iou[c]:>9.4f}{ft_iou[c]:>9.4f}{dlt:>+9.4f}{flag}")
    print(f"\nmIoU  : {base_test['mean_iou']:.4f} -> {ft_test['mean_iou']:.4f} "
          f"({ft_test['mean_iou'] - base_test['mean_iou']:+.4f})")
    print(f"FWIoU : {base_test['fwiou']:.4f} -> {ft_test['fwiou']:.4f} "
          f"({ft_test['fwiou'] - base_test['fwiou']:+.4f})")

    mi_up = ft_test["mean_iou"] >= base_test["mean_iou"]
    if mi_up and not drop_majority:
        print("\nVONIS: BERHASIL -- mIoU naik & kelas mayoritas tak turun. Layak dipromosikan "
              "(lihat cell promosi, set PROMOTE=True).")
    else:
        why = []
        if not mi_up: why.append("mIoU TIDAK naik di test")
        if drop_majority: why.append("mayoritas turun: " + ", ".join(CLASS_NAMES[c] for c in drop_majority))
        print("\nVONIS: BELUM memenuhi target (" + "; ".join(why) + "). Pertahankan baseline / "
              "tune ulang (FT_LR_HEAD/FT_LR_ENC, EPS, augmentasi). JANGAN promosikan.")

    save_json(ft_test, FT_DIR / "metrics_finetune.json")
    save_json({"model_key": MODEL_KEY + "_finetune", **MODEL_ARCH,
               "method": "full fine-tune (encoder dibuka, LR bertingkat) + median-frequency weighted focal+tversky+CE + sampler frequency-proportional",
               "dataset": BAHAN_DIR.name if BAHAN_DIR else "kaggle_attached_v4",
               "ft_lr_encoder": FT_LR_ENC, "ft_lr_head": FT_LR_HEAD,
               "encoder_frozen": False, "loss_class_weighted": True,
               "class_weights_median_freq": [round(float(w), 4) for w in class_w],
               "best_epoch_guarded": best_epoch,
               "baseline_test_miou": base_test["mean_iou"], "finetune_test_miou": ft_test["mean_iou"],
               "baseline_test_per_class_iou": base_test_iou, "finetune_test_per_class_iou": ft_iou,
               "per_class": ft_test["per_class"]}, FT_DIR / "summary_finetune.json")

    cm = np.array(ft_test["confusion_matrix"]); cmn = cm / cm.sum(axis=1, keepdims=True).clip(1)
    fig, axx = plt.subplots(figsize=(8, 6)); im = axx.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            axx.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=9,
                     color="white" if cmn[i, j] > 0.5 else "black")
    axx.set_xticks(range(N_CLASSES)); axx.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    axx.set_yticks(range(N_CLASSES)); axx.set_yticklabels(CLASS_NAMES)
    axx.set_xlabel("Predicted"); axx.set_ylabel("True"); axx.set_title("Confusion - fine-tune")
    fig.colorbar(im, ax=axx); fig.tight_layout()
    fig.savefig(FT_DIR / "confusion_matrix_finetune.png", dpi=120, bbox_inches="tight"); plt.show()

    try:
        export_to_onnx(ft_model, FT_DIR / "model_finetune.onnx",
                       in_channels=cfg["model"]["in_channels"], patch_size=cfg["inference"]["patch_size"])
        print("ONNX:", FT_DIR / "model_finetune.onnx")
    except Exception as e:
        print("ONNX dilewati:", e)
    print("Disimpan:", FT_DIR / "metrics_finetune.json", "| summary_finetune.json | confusion_matrix_finetune.png")


In [ ]:
# === (Opsional) Promosikan model fine-tune ke lokasi baseline -- HANYA bila VONIS berhasil ===
# Default OFF. Set PROMOTE=True secara SADAR setelah cek tabel BEFORE/AFTER. Baseline di-backup
# dulu (best_model_prefinetune_backup.pt) -> tetap reversible. Di Kaggle: tak ada mount Drive,
# jadi paket semua artefak FT_DIR jadi 1 zip (selalu, lepas dari PROMOTE) supaya tinggal
# download via panel Output Kaggle -> upload manual ke Drive di path DRIVE_TARGET_NAME.
PROMOTE = False
import shutil

if ENV not in ("colab", "lab"):
    zip_base = FT_DIR.parent / (FT_DIR.name + "_package")
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=FT_DIR)
    print("Paket siap diunduh:", zip_path)
    print("Langkah selanjutnya (manual, Kaggle tak ada mount Drive):")
    print("  1. Kaggle -> Save Version -> tunggu selesai -> buka tab 'Output' notebook ini.")
    print(f"  2. Unduh '{Path(zip_path).name}', lalu di Google Drive buat folder:")
    print(f"     Satria Data 3.0/{DRIVE_TARGET_NAME}")
    print("  3. Ekstrak isi zip ke folder itu.")
    print("  4. (Opsional) Kalau mau promosikan jadi baseline baru, jalankan ulang cell ini")
    print("     dgn ENV='colab'/'lab' (Drive ter-mount) setelah file ada di Drive.")
elif not PROMOTE:
    print("PROMOTE=False -- tidak menyalin apa pun. Baseline tetap aktif.")
    print("Hasil fine-tune tetap tersimpan lengkap di:", FT_DIR)
elif not FT_CKPT.exists():
    print("FT_CKPT tak ada -- tak ada yg dipromosikan.")
else:
    bak = BASE_DIR / "best_model_prefinetune_backup.pt"
    if not bak.exists():
        shutil.copy(BASE_DIR / "best_model.pt", bak); print("Backup baseline ->", bak)
    shutil.copy(FT_CKPT, BASE_DIR / "best_model.pt")
    shutil.copy(FT_DIR / "metrics_finetune.json", BASE_DIR / "metrics.json")
    if (FT_DIR / "model_finetune.onnx").exists():
        shutil.copy(FT_DIR / "model_finetune.onnx", BASE_DIR / "model.onnx")
    print("Dipromosikan ->", BASE_DIR, "(baseline sudah di-backup; reversible).")
